# 7. AU heatmaps by power threshold (raw → transform → three heatmaps per image)

Takes **date** and **frequency band** as input. Loads **raw parquet** from `work_dir/raw_data/<band>/`, runs the same transform logic with **1 MHz** frequency granularity, then plots **three images**, each with **three AU heatmaps**: (1) −90, −95, −100 dBm, (2) −105, −110, −115 dBm, (3) −120, −125, −130 dBm.

- **X axis:** full band frequency (GHz)
- **Y axis:** hour of day
- **Color:** airtime utilization (%) for detections above that threshold

In [146]:
import sys
from pathlib import Path
# Ensure repo root on path so "transform" can be imported from notebooks/
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from transform.transform import _statistics_per_hour, _date_hour_from_path, T_BIN_AU, T_INTER_CO

In [147]:
# Inputs: date and frequency band
date_input = "2026-02-03"   # YYYY-MM-DD or YYYY_MM_DD
band_input = "195MHz"     # e.g. 539MHz, 915MHz, 2441MHz, 3765MHz, 5500MHz

# Normalize date to YYYYMMDD for filename matching
date_str = date_input.replace("-", "_").strip()
if len(date_str) == 10 and date_str[4] == "_" and date_str[7] == "_":
    date_filter = date_str[:4] + date_str[5:7] + date_str[8:10]
elif len(date_str) == 8 and date_str.isdigit():
    date_filter = date_str
    date_str = f"{date_str[:4]}_{date_str[4:6]}_{date_str[6:8]}"
else:
    date_filter = date_str.replace("_", "")
    if len(date_filter) != 8:
        raise ValueError(f"Invalid date format: {date_input}. Use YYYY-MM-DD or YYYY_MM_DD.")

# Normalize band
band_str = str(band_input).strip()
if re.search(r"^\d+$", band_str):
    band_str = f"{band_str}MHz"
elif not re.search(r"MHz$", band_str, re.IGNORECASE):
    band_str = f"{band_str}MHz"

# Raw data path: work_dir/raw_data/<band>/
work_dir = Path("work_dir")
if not work_dir.exists():
    work_dir = Path("../work_dir")
raw_data = work_dir / "raw_data"
band_dir = raw_data / band_str

if not band_dir.exists():
    raise FileNotFoundError(f"Raw data folder not found: {band_dir}")

# Parquet files for this band; filter by date from filename
all_parquets = sorted(band_dir.glob("*.parquet"))
parquet_files = []
for f in all_parquets:
    dh = _date_hour_from_path(f)
    if dh and dh[0] == date_filter:
        parquet_files.append(f)

if not parquet_files:
    raise ValueError(f"No parquet files for date {date_input} and band {band_str} in {band_dir}")

print(f"Date: {date_input}  |  Band: {band_str}")
print(f"Found {len(parquet_files)} parquet file(s) for this date and band.")

Date: 2026-02-03  |  Band: 195MHz
Found 985 parquet file(s) for this date and band.


In [148]:
# Group by (date, hour) — same as transform
groups = {}  # (yyyymmdd, hour) -> list of paths
for f in parquet_files:
    dh = _date_hour_from_path(f)
    if dh is None:
        continue
    yyyymmdd, hour = dh
    key = (yyyymmdd, hour)
    groups.setdefault(key, []).append(f)

# Granularity (1 MHz for finer x-axis) and thresholds; AU uses 1-sec bins (MATLAB match)
cbw = 1e6
THRESHOLDS_ALL = [-90, -95, -100, -105, -110, -115, -120, -125, -130]

# For each threshold: hour -> (freq_centers_ghz, au_per_hour)
hour_data_by_threshold = {th: {} for th in THRESHOLDS_ALL}

for (yyyymmdd, hour), path_list in sorted(groups.items()):
    dfs = []
    for path in sorted(path_list):
        try:
            df = pd.read_parquet(path)
            if "timestamp" in df.columns and "freqs" in df.columns and "trace_max" in df.columns:
                dfs.append(df)
        except Exception:
            continue
    if not dfs:
        continue
    df_all = pd.concat(dfs, ignore_index=True)
    time_all = df_all["timestamp"].values
    unique_times = np.unique(time_all)
    n_denominator = min(len(unique_times), 3600)

    for curr_pow in THRESHOLDS_ALL:
        df_p = df_all[df_all["trace_max"] >= curr_pow]
        ts_all = df_p["timestamp"].values
        fr_all = df_p["freqs"].values
        if ts_all.size == 0 or fr_all.size == 0:
            hour_data_by_threshold[curr_pow][hour] = (np.array([]), np.array([]))
            continue
        co_ot, au_per_hour, ts_start, bin_t_co = _statistics_per_hour(
            ts_all, fr_all, cbw, T_BIN_AU, T_INTER_CO, n_denominator
        )
        f_min = np.floor(np.min(fr_all) / cbw) * cbw
        f_max = np.ceil(np.max(fr_all) / cbw) * cbw
        bin_f = np.arange(f_min, f_max + cbw * 0.5, cbw)
        freq_centers_ghz = (bin_f[:-1] + bin_f[1:]) / 2 / 1e9
        if len(au_per_hour) == len(freq_centers_ghz):
            hour_data_by_threshold[curr_pow][hour] = (freq_centers_ghz, au_per_hour)
        else:
            hour_data_by_threshold[curr_pow][hour] = (np.array([]), np.array([]))

hours_sorted = sorted(set(h for th in THRESHOLDS_ALL for h in hour_data_by_threshold[th]))
if not hours_sorted:
    raise ValueError("No valid (hour, threshold) data for this date and band.")

print(f"Hours with data: {hours_sorted}")
print(f"Thresholds: {len(THRESHOLDS_ALL)} values from {THRESHOLDS_ALL[0]} to {THRESHOLDS_ALL[-1]} dBm")

Hours with data: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 19, 20, 21, 22, 23]
Thresholds: 9 values from -90 to -130 dBm


In [149]:
# Full band extent (same as notebook 5)
BAND_EXTENT_MHZ = {
    "195MHz": (180, 220),
    "539MHz": (470, 610),
    "915MHz": (902, 928),
    "2441MHz": (2400, 2484),
    "3765MHz": (3550, 3980),
    "5500MHz": (5150, 5850),
}
f_min_mhz, f_max_mhz = BAND_EXTENT_MHZ.get(
    band_str,
    (3000, 4000),
)
channel_width_mhz = 1   # finer x-axis (1 MHz bins)
full_freq_centers_ghz = np.arange(
    (f_min_mhz + channel_width_mhz / 2) / 1000,
    f_max_mhz / 1000,
    channel_width_mhz / 1000,
)
n_full = len(full_freq_centers_ghz)
match_tol_ghz = (channel_width_mhz / 2) / 1000   # half of channel width (GHz)

hour_labels = [f"{h:02d}:00" for h in hours_sorted]
print(f"Full band: {f_min_mhz}–{f_max_mhz} MHz  |  {n_full} freq bins ({channel_width_mhz} MHz)")

Full band: 180–220 MHz  |  40 freq bins (1 MHz)


In [150]:
# Three figures, each with 3 heatmaps (3 thresholds per image)
THRESHOLD_GROUPS = [[-90, -95, -100], [-105, -110, -115], [-120, -125, -130]]
for group in THRESHOLD_GROUPS:
    matrices = {}
    for th in group:
        mat = np.zeros((len(hours_sorted), n_full))
        for i, hour in enumerate(hours_sorted):
            if hour not in hour_data_by_threshold[th]:
                continue
            freq_ghz, au_row = hour_data_by_threshold[th][hour]
            if freq_ghz.size == 0:
                continue
            for j, fc in enumerate(full_freq_centers_ghz):
                idx = np.argmin(np.abs(freq_ghz - fc))
                if np.abs(freq_ghz[idx] - fc) < match_tol_ghz and idx < len(au_row):
                    mat[i, j] = au_row[idx]
        matrices[th] = mat
    vmax_global = 0.0
    for th in group:
        m = matrices[th]
        if m.size:
            vmax_global = max(vmax_global, max(10.0, np.max(m) * 1.2))
    vmax_global = vmax_global or 100.0
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[f"≥ {th} dBm" for th in group],
        shared_yaxes=True, horizontal_spacing=0.06,
    )
    for col, th in enumerate(group, start=1):
        fig.add_trace(
            go.Heatmap(
                x=full_freq_centers_ghz,
                y=hour_labels,
                z=matrices[th],
                colorscale="Viridis", zmin=0, zmax=vmax_global,
                colorbar=dict(title="AU (%)"),
                hovertemplate="Freq: %{x:.3f} GHz<br>Time: %{y}<br>AU: %{z:.2f}%<extra></extra>",
            ), row=1, col=col,
        )
    fig.update_layout(
        title=f"Airtime utilization (%) — {date_input} — {band_str} — {group[0]} to {group[-1]} dBm",
        height=max(520, len(hours_sorted) * 28), showlegend=False,
    )
    fig.update_xaxes(title_text="Freq (GHz)", row=1, col=1)
    fig.update_xaxes(title_text="Freq (GHz)", row=1, col=2)
    fig.update_xaxes(title_text="Freq (GHz)", row=1, col=3)
    fig.update_yaxes(title_text="Time", row=1, col=1)
    fig.update_yaxes(showticklabels=True, row=1, col=2)
    fig.update_yaxes(showticklabels=True, row=1, col=3)
    fig.update_yaxes(autorange="reversed", row=1)
    fig.show()

In [151]:
# (All 9 heatmaps are generated in the cell above)

In [152]:
# (All 9 heatmaps are generated in the cell above)